# RAG Implement Instrcution

def rag_pipeline(query: str):
    
    1. retrieve docs
    
    2. pick top_k (e.g. 2)
    
    3. build context (join text)
    
    4. call llm
    
    5. return answer
    
    pass

In [1]:
# Register data
# 1. Instantiate Chroma DB and vectorize(register) the data (documents)

# Inference
# 1. Embed the user query
# 2. Filter metadata to reduce noise (or for narrow search space)
# 3. Hybrid retrieval: 
#   - Semantic vector database search with the query
#   - Keyword search 
# 4. Merge results
# 5. re-rank results (optional)
# 6. pick top_k
# 7. build context (prompt with context added)
# 8. call LLM 
# 9. generate answer

In [2]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from openai import OpenAI
from dotenv import load_dotenv
from typing import List
import os

In [3]:
load_dotenv()

True

In [4]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Register Vector DB

In [38]:
documents = [
    "Client A missed two deadlines last week.",
    "Client A requested a refund on Monday.",
    "Client B completed all tasks successfully.",
    "Client A has a high priority project ongoing.",

    "Client A delayed delivery on two tasks last week.",
    "Client A failed to meet two scheduled deadlines recently.",
    "Client A has been struggling to keep up with timelines.",
    "Client A reported issues affecting project progress.",
    "Client A escalated a concern regarding missed milestones.",
    
    "Client A submitted a refund request earlier this week.",
    "Client A asked for reimbursement on Monday.",
    "Client A raised a billing issue requiring refund processing.",
    
    "Client B finished all assigned work without delays.",
    "Client B successfully delivered all tasks on time.",
    "Client B met all project requirements with no issues.",
    
    "Client A is currently working on a high-priority project.",
    "Client A has an urgent project in progress.",
    "Client A is handling a critical task with high priority.",
    
    "Client C has just started onboarding this week.",
    "Client C is in the initial setup phase.",
    "Client C has not yet begun task execution.",
    
    "Client D missed one deadline but recovered quickly.",
    "Client D experienced minor delays but resolved them.",
    
    "Client E completed tasks ahead of schedule.",
    "Client E delivered results earlier than expected.",
    
    "Client A has multiple ongoing issues affecting delivery.",
    "Client A is under pressure due to tight deadlines.",
]

In [78]:
def create_or_load_collection(docs: List[str]):

    chroma = chromadb.PersistentClient(
        path="./chroma_db"
    ) 
    
    collection = chroma.get_or_create_collection(
        name="collection",
        embedding_function=OpenAIEmbeddingFunction(
            api_key=OPENAI_API_KEY,
            model_name="text-embedding-3-small"
        )
    )
      
    collection.add(
        documents=docs,
        ids=[str(i + 1) for i, doc in enumerate(documents)],
        metadatas=[{"source": f"doc{i}"} for i, doc in enumerate(documents)]
    )

    return collection

## Inference

In [79]:
def call_llm(user_input):
    
    llm = OpenAI(api_key=OPENAI_API_KEY)
    
    response = llm.chat.completions.create(
        model= "gpt-5-mini",
        messages= [
            {"role": "system", "content": "You are a helpful RAG LLM assistant"},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [80]:
def get_context(collection, query: str) -> List[str]:
    
    results = collection.query(
        query_texts=[query],
        n_results=5  # top_k 
    )

    print("RESULTS:\n", results)

    docs = results["documents"][0]

    return docs

DOCS:  ['Client A missed two deadlines last week.', 'Client A delayed delivery on two tasks last week.', 'Client A failed to meet two scheduled deadlines recently.', 'Client A reported issues affecting project progress.', 'Client B finished all assigned work without delays.']

In [81]:
def rerank_order(query: str, docs: List[str]):

    print("DOCS: ", docs)

    docs_text = "\n\n".join(
        [f"{i + 1}: {doc}" for i, doc in enumerate(docs)]
    )

    prompt = f"""
    You are reranking documents based on relevance to a query.

    Query: {query}

    Documents:
    {docs_text}

    Return ONLY the indices of the top 2 MOST relevant documents in order.
    Example: 2, 1 or 1, 2 
    
    """.strip()

    response = call_llm(prompt)
    print("RERANKING RESPONSE: ", response)

    indices = [int(i) - 1 for i in response.split(",")]

    reranked_docs = [docs[index] for index in indices]
    
    return reranked_docs
    
    

In [82]:
def rag_llm(context: str, query: str):

    print("QUERY: ", query)
    print("CONTEXT: ", context)
    user_input = f"""
    You are a question-answering system.

    Answer the user question using ONLY the provided context.
    You are allowed to rephrase or summarize the context to aswer the question (in a polished way)

    if the answer is not presnet in the context, say "I don't know".
        
    Context:
    {context}

    Question:
    {query}
    
    Your Answer: 
    
    """.strip()

    response = call_llm(user_input)  

    return response

In [85]:
def rag_pipeline(query: str):
    
    # Create or load collection (Chroma)
    collection = create_or_load_collection(documents)

    # Retrieve chunks on top_k = n
    top_k_docs = get_context(collection, query)
    
    # Re-ranking it for more accurate context results
    reranked_context = rerank_order(query, top_k_docs)

    # Generate the final response
    response = rag_llm(reranked_context, query)

    return response

In [86]:
print("FINAL RESPONSE:\n", rag_pipeline("what happeded to Client A last week?"))

RESULTS:
 {'ids': [['1', '5', '6', '8', '13']], 'embeddings': None, 'documents': [['Client A missed two deadlines last week.', 'Client A delayed delivery on two tasks last week.', 'Client A failed to meet two scheduled deadlines recently.', 'Client A reported issues affecting project progress.', 'Client B finished all assigned work without delays.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, {'source': 'doc4'}, {'source': 'doc5'}, {'source': 'doc7'}, {'source': 'doc12'}]], 'distances': [[0.4184814691543579, 0.4403477907180786, 0.450259268283844, 0.455829918384552, 0.46312832832336426]]}
DOCS:  ['Client A missed two deadlines last week.', 'Client A delayed delivery on two tasks last week.', 'Client A failed to meet two scheduled deadlines recently.', 'Client A reported issues affecting project progress.', 'Client B finished all assigned work without delays.']
RERANKING RESPONSE:  1, 2
QUERY:  what happeded to Client A last wee